<a href="https://colab.research.google.com/github/Amt2525/classificador_de_arquivos/blob/main/analise_imagens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por
!pip install -q pytesseract opencv-python-headless pandas

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [12]:

import os

# Pasta onde estão as imagens
PASTA = "/content/arquivos"

# Categorias do projeto
categorias = ["rg_antigos", "rg_novos", "invalidos"]

for categoria in categorias:
    caminho = os.path.join(PASTA, categoria)

    if os.path.isdir(caminho):
        arquivos = [
            nome for nome in os.listdir(caminho)
            if nome.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            )
        ]

        print(f"{categoria}: {len(arquivos)} imagens")
    else:
        print(f"Pasta não encontrada: {caminho}")


rg_antigos: 100 imagens
rg_novos: 100 imagens
invalidos: 100 imagens


In [13]:
import os
import cv2
import pandas as pd

# Caminho principal das imagens
PASTA = "/content/arquivos"

# Categorias dos documentos
categorias = ["rg_antigos", "rg_novos", "invalidos"]

# Extensões de imagens aceitas
extensoes = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for categoria in categorias:
    caminho = os.path.join(PASTA, categoria)

    if not os.path.isdir(caminho):
        print(f"Pasta não encontrada: {caminho}")
        continue

    imagens = [
        arquivo for arquivo in os.listdir(caminho)
        if arquivo.lower().endswith(extensoes)
    ]

    print(f"{categoria}: {len(imagens)} imagens encontradas")

rg_antigos: 100 imagens encontradas
rg_novos: 100 imagens encontradas
invalidos: 100 imagens encontradas


In [14]:
import os
import cv2
import pandas as pd
import pytesseract

# Pasta principal das imagens
PASTA = "/content/arquivos"

categorias = ["rg_antigos", "rg_novos", "invalidos"]
extensoes = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

dados = []

for categoria in categorias:
    pasta_categoria = os.path.join(PASTA, categoria)

    if not os.path.isdir(pasta_categoria):
        print(f"Pasta não encontrada: {pasta_categoria}")
        continue

    for arquivo in os.listdir(pasta_categoria):
        if not arquivo.lower().endswith(extensoes):
            continue

        caminho = os.path.join(pasta_categoria, arquivo)
        imagem = cv2.imread(caminho)

        if imagem is None:
            print(f"Não foi possível abrir: {arquivo}")
            continue

        altura, largura = imagem.shape[:2]

        # Extrair texto da imagem
        texto = pytesseract.image_to_string(imagem, lang="por")
        texto = texto.upper()

        # Registrar as características encontradas
        dados.append({
            "arquivo": arquivo,
            "classe": categoria,
            "largura": largura,
            "altura": altura,
            "quantidade_texto": len(texto.strip()),
            "tem_texto_rg": int("RG" in texto),
            "tem_texto_cpf": int("CPF" in texto),
            "texto_republica": int("REPUBLICA" in texto),
            "tem_qr_code": int(
                cv2.QRCodeDetector().detect(imagem)[0]
            )
        })

# Criar a tabela
df = pd.DataFrame(dados)

# Salvar o CSV
df.to_csv(
    "/content/analise_documentos.csv",
    index=False,
    encoding="utf-8"
)

print("Análise concluída!")
print(f"Total de imagens analisadas: {len(df)}")
display(df.head())

Análise concluída!
Total de imagens analisadas: 300


,arquivo,classe,largura,altura,quantidade_texto,tem_texto_rg,tem_texto_cpf,texto_republica,tem_qr_code
0,00022280_in.jpg,rg_antigos,963,578,152,0,0,0,0
1,00022192_in.jpg,rg_antigos,1225,871,110,0,0,0,0
2,00022271_in.jpg,rg_antigos,865,604,35,0,0,0,0
3,00022243_in.jpg,rg_antigos,1008,708,0,0,0,0,0
4,00022226_in.jpg,rg_antigos,686,483,69,0,0,0,0
